# 13 — Execution and comparison

**The concept:** running a model costs something, and an engineer asked to consent to a run deserves
a **number**, not a spinner. `run_pipeline` executes a model file in a child process, under a
mandatory wall clock, and returns typed results.

It is deliberately a *ladder*: the cheapest rung proves the code executes **and** measures the unit
cost, so every estimate above it falls out of it.

In [1]:
import pathlib, tempfile
from smartmdao.mcp.handlers import run_pipeline, compare_runs

workspace = pathlib.Path(tempfile.mkdtemp())
model_a = workspace / "faithful.py"
model_a_source = '''# A small MDA, written the faithful way: converge on y2 ALONE.
from smartmdao import Pipeline, HybridSolver


def build_pipeline() -> Pipeline:
    pipeline = Pipeline(solver=HybridSolver(target_var="y2", tolerance=1e-6))

    @pipeline.step(outputs=["y1"])
    def discipline_1(z: float, y2: float) -> float:
        return z**2 - 0.2 * y2

    @pipeline.step(outputs=["y2"])
    def discipline_2(y1: float) -> float:
        return abs(y1) ** 0.5

    return pipeline
'''
model_a.write_text(model_a_source)

print(model_a.read_text())

# A small MDA, written the faithful way: converge on y2 ALONE.
from smartmdao import Pipeline, HybridSolver


def build_pipeline() -> Pipeline:
    pipeline = Pipeline(solver=HybridSolver(target_var="y2", tolerance=1e-6))

    @pipeline.step(outputs=["y1"])
    def discipline_1(z: float, y2: float) -> float:
        return z**2 - 0.2 * y2

    @pipeline.step(outputs=["y2"])
    def discipline_2(y1: float) -> float:
        return abs(y1) ** 0.5

    return pipeline



## The `smoke` rung — one sweep, and a cost estimate

`rung="smoke"` is the **default**, deliberately: an unbounded run should never be the thing that
happens by accident.

In [2]:
smoke = run_pipeline(str(model_a), inputs={"z": 2.0, "y2": 1.0})

print("rung:      ", smoke["rung"])
print("converged: ", smoke.get("converged"))
for key in ("elapsed_seconds", "cost_estimate"):
    if key in smoke:
        print(f"{key}:", smoke[key])
print()
print("state:", {k: round(v, 6) for k, v in smoke["state"].items() if isinstance(v, float)})

rung:       smoke
converged:  False
elapsed_seconds: 0.0005
cost_estimate: {'one_sweep_seconds': 0.0005, 'budgeted_worst_case_seconds': 0.013, 'full_worst_case_seconds': 0.05, 'note': 'one sweep took 0.0005s; a full run may need up to 100 sweeps, so budget up to 0.05s. Quote this before running it.'}

state: {'z': 2.0, 'y2': 1.949359, 'y1': 3.8}


One sweep proves the file imports, the graph wires up, and every discipline can be called. It
does **not** prove the answer has converged — that is what the next rung is for.

## `budgeted` and `full`

`budgeted` caps the number of sweeps; `full` runs until it converges or the wall clock stops it.
Both are child processes with a mandatory timeout, so a non-converging model is killed and
**reported as such** rather than hanging.

In [3]:
budgeted = run_pipeline(str(model_a), inputs={"z": 2.0, "y2": 1.0},
                        rung="budgeted", budget_sweeps=50, timeout_seconds=30)
full = run_pipeline(str(model_a), inputs={"z": 2.0, "y2": 1.0},
                    rung="full", timeout_seconds=30)

for label, outcome in (("budgeted", budgeted), ("full", full)):
    print(f"{label:9} converged={outcome.get('converged')} "
          f"y1={outcome['state']['y1']:.6f} y2={outcome['state']['y2']:.6f}")

budgeted  converged=True y1=3.619500 y2=1.902498
full      converged=True y1=3.619500 y2=1.902498


## Inputs are recovered from the file

If you pass no inputs, the loader reads the file's **own** `run(...)` call statically — AST only,
nothing executed — so `run_pipeline(path)` behaves the way running the script does. The response
says where each value came from, so a report can state it rather than assert it.

In [4]:
# A module-level pipeline that calls run() itself - the shape a script has.
with_inputs = workspace / "with_inputs.py"
with_inputs.write_text('''from smartmdao import Pipeline, HybridSolver

pipeline = Pipeline(solver=HybridSolver(target_var="y2", tolerance=1e-6))

@pipeline.step(outputs=["y1"])
def discipline_1(z: float, y2: float) -> float:
    return z**2 - 0.2 * y2

@pipeline.step(outputs=["y2"])
def discipline_2(y1: float) -> float:
    return abs(y1) ** 0.5

result = pipeline.run(z=3.0, y2=1.0)
''')

recovered = run_pipeline(str(with_inputs))
print("inputs_used:", recovered["inputs_used"])
print("state:      ", {k: round(v, 6) for k, v in recovered["state"].items()})

inputs_used: {'supplied': [], 'found_in_source': ['y2', 'z']}
state:       {'z': 3.0, 'y2': 2.966479, 'y1': 8.8}


## A crash is a result, not a disappearance

The subprocess boundary means a discipline that raises comes back as typed information instead of
taking your session with it.

**What it does not buy is safety.** The agent driving this has a shell and will run the file itself
if refused. The subprocess buys a hard kill, typed results and crash isolation — nothing more, and
the project is explicit about not letting that claim drift.

In [5]:
crashing = workspace / "crashes.py"
crashing.write_text(
    "from smartmdao import Pipeline\n\n"
    "def build_pipeline() -> Pipeline:\n"
    "    p = Pipeline()\n\n"
    "    @p.step(outputs=['y'])\n"
    "    def boom(x: float) -> float:\n"
    "        raise ValueError('the discipline failed')\n\n"
    "    return p\n"
)

outcome = run_pipeline(str(crashing), inputs={"x": 1.0})
print({k: v for k, v in outcome.items() if k in ("ok", "error", "error_type", "rung")})

{'ok': False, 'error': "RuntimeError: Error executing step 'boom': the discipline failed", 'rung': 'smoke'}


## `compare_runs` — proving a translation kept the answer

Converting hand-written code to SmartMDAO is a stated use case, and its failure mode is the worst
kind: the translation looks cleaner, so it gets trusted, and it quietly returns a different number.

`compare_runs` runs **both** files on the **same** inputs and diffs the resulting state.

In [6]:
model_b = workspace / "idiomatic.py"
model_b.write_text('''# The same model, translated idiomatically: no target_var.
from smartmdao import Pipeline, HybridSolver


def build_pipeline() -> Pipeline:
    pipeline = Pipeline(solver=HybridSolver(tolerance=1e-6))

    @pipeline.step(outputs=["y1"])
    def discipline_1(z: float, y2: float) -> float:
        return z**2 - 0.2 * y2

    @pipeline.step(outputs=["y2"])
    def discipline_2(y1: float) -> float:
        return abs(y1) ** 0.5

    return pipeline
''')

same = compare_runs(str(model_a), str(model_b), inputs={"z": 2.0, "y2": 1.0})
print("match:", same.get("match"))
print("differences:", same.get("differences"))

match: True
differences: []


Now a translation that **did** change the answer — it watches a different variable and settles
with a looser tolerance. Both files are perfectly well-formed, both converge, and nothing that reads
structure could catch it.

In [7]:
model_drift = workspace / "drifted.py"
model_drift.write_text('''# A translation that changed the answer, and says nothing about it:
# it watches y1 instead of y2, and settles far looser.
from smartmdao import Pipeline, HybridSolver


def build_pipeline() -> Pipeline:
    pipeline = Pipeline(solver=HybridSolver(target_var="y1", tolerance=5.0))

    @pipeline.step(outputs=["y1"])
    def discipline_1(z: float, y2: float) -> float:
        return z**2 - 0.2 * y2

    @pipeline.step(outputs=["y2"])
    def discipline_2(y1: float) -> float:
        return abs(y1) ** 0.5

    return pipeline
''')

from smartmdao import validate
from smartmdao.mcp.loader import load_pipeline

for path in (model_a, model_drift):
    loaded = load_pipeline(str(path))
    print(f"{path.name:14} validate() -> {validate(loaded.pipeline, inputs=['z', 'y2']) or 'clean'}")

print()
drift = compare_runs(str(model_a), str(model_drift), inputs={"z": 2.0, "y2": 1.0})
print("match:", drift.get("match"))
for difference in drift.get("differences", []):
    print(" ", difference)

faithful.py    validate() -> clean
drifted.py     validate() -> clean



match: False
  {'variable': 'y1', 'a': 3.6195002405275636, 'b': 3.6101282262076415, 'absolute': 0.009372014319922073, 'relative': 0.0025893116997157723}
  {'variable': 'y2', 'a': 1.9024984206373374, 'b': 1.9000337434392163, 'absolute': 0.002464677198121157, 'relative': 0.0012954950035099053}


Both clean, both converged, different answers. That is the entire argument for execution
tooling: **structure cannot tell you the answer changed.**

Numbers compare within a tolerance, so a different iteration count is absorbed. Anything
non-numeric compares exactly, because there is no "nearly" for a frozenset of decisions. And a
different *destination* counts too — close numbers are not a match when one side never converged.

---

**Next:** [14 — Pipeline discovery and the MCP connector](14-pipeline-discovery-and-mcp.ipynb).